# Data Visualization – seq\_ue\_calibration

Loads the pre-computed calibration data from `resources/prepared_data.pkl`
(generated by `analysis_prepare_data`) and produces all figures and LaTeX tables.

**Run order**: `analysis_prepare_data` must be executed first to create the pickle.

In [1]:
%load_ext autoreload
%autoreload 2

## Imports

In [2]:
import json
import os
import pickle
from collections import defaultdict
from functools import partial
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [3]:
from datasets_exp2 import (
    arithmetic_datasets,
    arithmetic_datasets_dict,
    datasets,
    datasets_combined,
    mc_datasets,
    mc_datasets_dict,
)

from util.config import apply_matplotlib_defaults
from util.plot_empty import plot_empty

In [4]:
# Shared configuration: model list, UQ method list, and output paths
from analysis_config import (
    models,
    uq_methods,
    RESOURCES_DIR,
    FIGURES_DIR,
    TABLES_DIR,
    PREPARED_DATA_PATH,
)

In [5]:
# Timestamped print helper (used throughout to track long-running loops)
from analysis_utils.misc_utils import print_with_time

# Colour blending utility (lighten/darken named colours; used for bar-chart fill colours)
from analysis_utils.color_utils import adjust_color, scale_fonts

# Counts unique answers per question id; used for the arithmetic histogram
from analysis_utils.dataframe_utils import unique_count_distribution

# Calibration subplot renderer, stats-table renderer, relplot wrapper, pre-built grid assembler
# and legacy grid builders (used as-is from analysis.md for backward compat)
from analysis_utils.calibration_plot_helpers import (
    plot_calibration_subplot,
    plot_calibration_stats_table,
    plot_relplot_subplot,
    assemble_cal_grid,
    build_calibration_grid_by_model,
    build_calibration_grid_by_uq_method,
)

# LaTeX table builders: accuracy comparison, response-length stats, scalar metric table
from analysis_utils.latex_utils import (
    make_accuracy_table_latex,
    make_length_table_latex,
    make_scalar_table_latex,
)

# Donut-chart grid for verbalized confidence distributions; bar-chart grid for P(True) bucket counts
from analysis_utils.distribution_plots import plot_donut_row_with_other, plot_bucket_distribution

## Setup

In [6]:
apply_matplotlib_defaults()

for d in [RESOURCES_DIR, FIGURES_DIR, TABLES_DIR]:
    os.makedirs(d, exist_ok=True)
os.makedirs(FIGURES_DIR / "by_model",     exist_ok=True)
os.makedirs(FIGURES_DIR / "by_uq_method", exist_ok=True)

# Light fill colours for bar charts, keyed by model type
color = {
    "instruct":  adjust_color("tab:blue",  0.7, lighten=True),
    "reasoning": adjust_color("tab:green", 0.7, lighten=True),
}

## Load prepared data

In [7]:
with open(PREPARED_DATA_PATH, "rb") as fh:
    prepared_data = pickle.load(fh)

print("Loaded prepared data for datasets:", list(prepared_data.keys()))

FileNotFoundError: [Errno 2] No such file or directory: 'resources\\prepared_data.pkl'

## Reshape cal_data for table utilities

`make_cal_metric_table_latex` expects `cal_data[model_id][dataset_id][uq_method_id]`.
This is a simple 3-level restructure from the pickle's `[dataset_id][model_id]["cal"]`.

In [ ]:
cal_data = {
    model["id"]: {
        ds_id: prepared_data[ds_id][model["id"]]["cal"]
        for ds_id in prepared_data
    }
    for model in models
}

## Load accuracy tables (produced by analysis_prepare_data)

In [ ]:
with open(RESOURCES_DIR / "accuracy_per_ds_per_model_mc.json", encoding="utf-8") as f:
    accuracy_per_ds_per_model_mc = json.load(f)

with open(RESOURCES_DIR / "accuracy_per_ds_per_model_arithmetic.json", encoding="utf-8") as f:
    accuracy_per_ds_per_model_arithmetic = json.load(f)

## Derived data

Quick aggregations from the prepared DataFrames — these are cheap enough to
compute on-the-fly in the visualization notebook rather than in the pickle.

In [ ]:
# verbalized_stats[model_id] = pd.Series of verbalized confidence value counts,
# summed across all datasets.
verbalized_stats = {}
for model in models:
    value_counts = None
    for dataset in datasets:
        df = prepared_data[dataset["id"]][model["id"]]["df"]
        vc = df["verbalized"].value_counts()
        value_counts = vc if value_counts is None else value_counts.add(vc, fill_value=0)
    verbalized_stats[model["id"]] = value_counts

In [ ]:
# ptrue_bucket_counts[model_id] = summed bucket_counts array across all datasets
ptrue_bucket_counts = {}
for model in models:
    total = np.zeros(15, dtype=int)
    for dataset in datasets:
        cal_entry = prepared_data[dataset["id"]][model["id"]]["cal"].get("p_true")
        if cal_entry is not None:
            total += np.array(cal_entry["bucket_counts"], dtype=int)
    ptrue_bucket_counts[model["id"]] = total

In [ ]:
# length_metadata[dataset_id][model_id] = {"sum", "mean", "std"} for answer_token_len
length_metadata = {}
for dataset in datasets:
    ds_id = dataset["id"]
    length_metadata[ds_id] = {}
    for model in models:
        lengths = prepared_data[ds_id][model["id"]]["df"]["answer_token_len"]
        length_metadata[ds_id][model["id"]] = {
            "sum":  int(lengths.sum()),
            "mean": float(lengths.mean()),
            "std":  float(lengths.std()),
        }

## Pre-build subplot callables

For each (dataset × model × uq_method) triple, create three `functools.partial`
callables that bind data and parameters at construction time:

- `"cal"`: binned calibration curve subplot
- `"relplot"`: kernel-smoothed reliability diagram (uses pre-computed diagram from pickle)
- `"table"`: stats table, including smECE from the relplot diagram

Storing callables instead of rendered figures keeps memory usage low while
making grid assembly trivial — each grid just maps callables onto Axes.

In [ ]:
subplots = {}

for dataset in datasets:
    ds_id = dataset["id"]
    subplots[ds_id] = {}

    for model in models:
        mid = model["id"]
        entry = prepared_data[ds_id][mid]
        subplots[ds_id][mid] = {}

        for uq_method in uq_methods:
            uq_id = uq_method["id"]
            cal_entry = entry["cal"].get(uq_id)

            if cal_entry is None:
                subplots[ds_id][mid][uq_id] = {
                    "cal":     plot_empty,
                    "relplot": plot_empty,
                    "table":   plot_empty,
                }
            else:
                smooth_ece         = cal_entry["relplot_diagram"]["ce"]
                smooth_ece_ci_width = cal_entry["relplot_diagram"].get("ce_ci_width")
                subplots[ds_id][mid][uq_id] = {
                    "cal":     partial(plot_calibration_subplot,
                                       data_item=cal_entry, model_type=model["type"]),
                    "relplot": partial(plot_relplot_subplot,
                                       data_item=cal_entry, model_type=model["type"]),
                    "table":   partial(plot_calibration_stats_table,
                                       data_item=cal_entry,
                                       total_items=entry["total_items"],
                                       invalid_answers=entry["invalid_answers"],
                                       smooth_ece=smooth_ece,
                                       smooth_ece_ci_width=smooth_ece_ci_width),
                }

print("Pre-built subplots for", sum(len(v) for v in subplots.values()), "model/dataset pairs.")

## Helpers: build calibration grids from pre-built subplots

In [ ]:
def build_grid_by_model(model, datasets_subset, subplot_type, with_table, with_title=True, **kwargs):
    """Assemble a calibration grid (datasets × uq_methods) for one model.

    Slices from the pre-built ``subplots`` dict and calls ``assemble_cal_grid``.

    Args:
        model: Model dict with keys ``"id"``, ``"shortname"``, ``"type"``.
        datasets_subset: Ordered list of dataset dicts to use as rows.
        subplot_type: ``"cal"`` for binned calibration, ``"relplot"`` for
            kernel-smoothed reliability diagrams.
        with_table: When ``True``, include the stats-table row below each
            calibration row.
        with_title: When ``True``, add a figure-level title with the model name.
        **kwargs: Forwarded to ``assemble_cal_grid``.

    Returns:
        matplotlib.figure.Figure
    """
    subplot_fns = [
        [subplots[ds["id"]][model["id"]][uq["id"]][subplot_type] for uq in uq_methods]
        for ds in datasets_subset
    ]
    table_fns = [
        [subplots[ds["id"]][model["id"]][uq["id"]]["table"] for uq in uq_methods]
        for ds in datasets_subset
    ] if with_table else None

    row_titles = [ds["id"].replace("_", "-") for ds in datasets_subset]
    col_titles = [uq["label"] for uq in uq_methods]
    title = f"Calibration Plots for Model {model['shortname']}" if with_title else None
    subplot_aspect = 1 if subplot_type == "cal" else None

    return assemble_cal_grid(subplot_fns, row_titles, col_titles, table_fns,
                              plot_title=title, subplot_aspect=subplot_aspect, **kwargs)


def build_grid_by_uq_method(uq_method, datasets_subset, models_subset, subplot_type, with_table,
                             with_title=True, **kwargs):
    """Assemble a calibration grid (datasets × models) for one UQ method.

    Slices from the pre-built ``subplots`` dict and calls ``assemble_cal_grid``.

    Args:
        uq_method: UQ method dict with keys ``"id"`` and ``"label"``.
        datasets_subset: Ordered list of dataset dicts to use as rows.
        models_subset: Ordered list of model dicts to use as columns.
        subplot_type: ``"cal"`` or ``"relplot"``.
        with_table: When ``True``, include stats-table rows.
        with_title: When ``True``, add a figure-level title with the UQ method name.
        **kwargs: Forwarded to ``assemble_cal_grid``.

    Returns:
        matplotlib.figure.Figure
    """
    uq_id = uq_method["id"]
    subplot_fns = [
        [subplots[ds["id"]][m["id"]][uq_id][subplot_type] for m in models_subset]
        for ds in datasets_subset
    ]
    table_fns = [
        [subplots[ds["id"]][m["id"]][uq_id]["table"] for m in models_subset]
        for ds in datasets_subset
    ] if with_table else None

    row_titles = [ds["id"].replace("_", "-") for ds in datasets_subset]
    col_titles = [m["shortname"] for m in models_subset]
    title = f"Calibration Plots for Uncertainty Metric {uq_method['label']}" if with_title else None
    subplot_aspect = 1 if subplot_type == "cal" else None

    return assemble_cal_grid(subplot_fns, row_titles, col_titles, table_fns,
                              plot_title=title, subplot_aspect=subplot_aspect, **kwargs)

## Arithmetic Answer Count Histogram

In [ ]:
nrows = len(arithmetic_datasets)
ncols = len(models)
fig, axes = plt.subplots(
    nrows, ncols,
    figsize=(ncols * 3.5, nrows * 3.05),
    gridspec_kw={"width_ratios": [1] * ncols, "height_ratios": [1] * nrows},
    squeeze=False,
)

for i, dataset in enumerate(arithmetic_datasets):
    axes[i, 0].text(
        -0.3, 0.5, dataset["label"], ha="center", va="center",
        rotation="vertical", fontsize=16, fontweight="bold",
        transform=axes[i, 0].transAxes,
    )

for i, dataset in enumerate(arithmetic_datasets):
    for j, model in enumerate(models):
        ax = axes[i, j]
        df = prepared_data[dataset["id"]][model["id"]]["df"]
        count_col = "cluster_id" if dataset["id"] == "SciBench" else "extracted_number"
        count_dict = unique_count_distribution(df, count_col)

        x_vals = list(range(1, 11))
        y_vals = [count_dict.get(x, 0) for x in x_vals]
        mean = np.average(list(count_dict.keys()), weights=list(count_dict.values()))

        ax.bar(x_vals, y_vals, color=color[model["type"]], edgecolor="black")
        ax.set_xlim(0.5, 10.5)
        ax.set_xticks(x_vals)
        if i == len(arithmetic_datasets) - 1:
            ax.set_xlabel("Count of Different Arithmetic Results", fontsize=12)
        if j == 0:
            ax.set_ylabel("Count of Dataset Items", fontsize=12)
        if i == 0:
            ax.annotate(
                model["shortname"], xy=(0.5, 1.05), xycoords="axes fraction",
                ha="center", va="bottom", fontsize=16, fontweight="bold",
            )

plt.tight_layout()
for ext in ["svg", "pdf", "png"]:
    plt.savefig(FIGURES_DIR / f"arithmetic_answer_count.{ext}", bbox_inches="tight")
plt.show()

## Accuracy Tables

In [ ]:
model_ids = [m["id"] for m in models]

latex_table = make_accuracy_table_latex(
    accuracy_per_ds_per_model_mc,
    {
        "accuracy":           "Accuracy",
        "precision":          "Precision",
        "recall":             "Recall",
        "f1":                 "F1-Score",
        "accuracy_choices":   "Accuracy across Choices",
        "accuracy_questions": "Accuracy across Questions",
    },
    model_ids,
    caption=(
        r"\caption[Accuracy, Precision, Recall, F1, and Consistency Metrics Across Models and MC Datasets]"
        r"{\textbf{Comparison of Accuracy, Precision, Recall, F1-Score, and Consistency Metrics Across Models "
        r"and Multiple‐Choice Datasets.} Shown are base accuracy, precision, recall, F1-score, accuracy "
        r"across choices (proportion of correctly classified options over ten generations), and accuracy "
        r"across questions (proportion of questions with all four options correct over ten generations) "
        r"for each model–dataset pair.}"
    ),
)
with open(TABLES_DIR / "accuracy_mc_datasets.tex", "w", encoding="utf-8") as f:
    f.write(latex_table)

In [ ]:
latex_table = make_accuracy_table_latex(
    accuracy_per_ds_per_model_arithmetic,
    {
        "accuracy":                    "Accuracy",
        "accuracy_questions":          "Accuracy across Questions",
        "different_answer_count_mean": "Mean of Different Answers across Iterations",
    },
    model_ids,
    caption=(
        r"\caption[Accuracy, Consistency, and Answer Variability Across Arithmetic Datasets]"
        r"{\textbf{Comparison of Accuracy, Question‐Level Consistency, and Answer Variability Across "
        r"Arithmetic Datasets.} Metrics include base accuracy, the proportion of questions with all "
        r"correct answers over ten runs, and the mean number of distinct answers produced across "
        r"iterations for each model–dataset pair.}"
    ),
)
with open(TABLES_DIR / "accuracy_arithmetic_datasets.tex", "w", encoding="utf-8") as f:
    f.write(latex_table)

## Calibration Plots by Model

One grid per model: rows = datasets, columns = UQ methods.

In [ ]:
for model in models:
    for with_table in [True, False]:
        print_with_time(f"Creating bucket-cal grid for {model['name']} (with_table={with_table}) ...")
        suffix = "_with_table" if with_table else ""
        fig = build_grid_by_model(model, datasets, "cal", with_table)
        for ext in ["svg", "png"]:
            fig.savefig(FIGURES_DIR / f"by_model/{model['id']}_calibration_plots{suffix}.{ext}",
                        bbox_inches="tight")
        plt.close(fig)

In [ ]:
# Short version: MMLU + GSM8K only, no title
short_datasets = [d for d in datasets if d["id"] in ("MMLU", "GSM8K")]
for model in models:
    fig = build_grid_by_model(model, short_datasets, "cal", with_table=False, with_title=False)
    for ext in ["svg", "png"]:
        fig.savefig(FIGURES_DIR / f"by_model/short_{model['id']}_calibration_plots_short.{ext}",
                    bbox_inches="tight")
    plt.close(fig)

## Calibration Plots by Model (relplot)

Same grid layout but each subplot is a kernel-smoothed reliability diagram.

In [ ]:
for model in models:
    for with_table in [True, False]:
        print_with_time(f"Creating relplot grid for {model['name']} (with_table={with_table}) ...")
        suffix = "_with_table" if with_table else ""
        fig = build_grid_by_model(model, datasets, "relplot", with_table)
        for ext in ["svg", "png"]:
            fig.savefig(FIGURES_DIR / f"by_model/{model['id']}_calibration_plots{suffix}_relplot.{ext}",
                        bbox_inches="tight")
        plt.close(fig)

# Restore matplotlib defaults after relplot rendering (relplot's set_default_style()
# calls mpl.rc_file_defaults() as a side-effect; individual calls are guarded in
# plot_relplot_subplot, but this is a belt-and-suspenders reset for good measure).
apply_matplotlib_defaults()

In [ ]:
for model in models:
    fig = build_grid_by_model(model, short_datasets, "relplot", with_table=False, with_title=False)
    for ext in ["svg", "png"]:
        fig.savefig(FIGURES_DIR / f"by_model/short_{model['id']}_calibration_plots_short_relplot.{ext}",
                    bbox_inches="tight")
    plt.close(fig)
apply_matplotlib_defaults()

## Calibration Plots by UQ Method

One grid per UQ method: rows = datasets, columns = models.

In [ ]:
for uq_method in uq_methods:
    for with_table in [True, False]:
        print_with_time(f"Creating bucket-cal grid for {uq_method['label']} (with_table={with_table}) ...")
        label_safe = uq_method["label"].replace(" ", "_")
        suffix = "_with_table" if with_table else ""
        fig = build_grid_by_uq_method(uq_method, datasets, models, "cal", with_table)
        for ext in ["svg", "png"]:
            fig.savefig(
                FIGURES_DIR / f"by_uq_method/uq_method_{label_safe}_calibration_plots{suffix}.{ext}",
                bbox_inches="tight",
            )
        plt.close(fig)

## Calibration Plots by UQ Method (relplot)

In [ ]:
for uq_method in uq_methods:
    for with_table in [True, False]:
        print_with_time(f"Creating relplot grid for {uq_method['label']} (with_table={with_table}) ...")
        label_safe = uq_method["label"].replace(" ", "_")
        suffix = "_with_table" if with_table else ""
        fig = build_grid_by_uq_method(uq_method, datasets, models, "relplot", with_table)
        for ext in ["svg", "png"]:
            fig.savefig(
                FIGURES_DIR / f"by_uq_method/uq_method_{label_safe}_calibration_plots{suffix}_relplot.{ext}",
                bbox_inches="tight",
            )
        plt.close(fig)
apply_matplotlib_defaults()

## Response Lengths Table

In [ ]:
latex_table = make_length_table_latex(
    length_metadata,
    model_ids=[m["id"] for m in models],
    dataset_ids=list(datasets_combined),
)
with open(TABLES_DIR / "response_lengths.tex", "w", encoding="utf-8") as f:
    f.write(latex_table)
print(latex_table)

## ECE, Entropy, and AUROC Tables

`make_cal_metric_table_latex` reads from the reshaped `cal_data` dict.

In [ ]:
def make_cal_metric_table_latex(cal_data, models, datasets, uq_method, prop, caption):
    """Build a scalar LaTeX table for one calibration property extracted from cal_data.

    Extracts ``cal_data[model_id][dataset_id][uq_method_id][prop]`` into a flat
    values dict and delegates table formatting to ``make_scalar_table_latex``.

    Args:
        cal_data: Nested dict ``cal_data[model_id][dataset_id][uq_method_id]``.
        models: List of model dicts with ``"id"`` key.
        datasets: Iterable of dataset IDs (used as row keys).
        uq_method: Metric dict with ``"id"`` key.
        prop: Property key to extract (e.g. ``"ece"``, ``"auroc"``).
        caption: Table caption string.

    Returns:
        str: Complete LaTeX table code.
    """
    model_ids    = [m["id"] for m in models]
    uq_method_id = uq_method["id"]
    values = {
        ds_id: {
            mid: (cal_data[mid][ds_id].get(uq_method_id) or {}).get(prop, float("nan"))
            for mid in model_ids
        }
        for ds_id in datasets
    }
    return make_scalar_table_latex(list(datasets), model_ids, values, caption)

In [ ]:
for uq_method in uq_methods:
    latex_table = make_cal_metric_table_latex(
        cal_data, models, datasets_combined, uq_method=uq_method,
        prop="ece",
        caption=f"ECE by Model and Dataset for {uq_method['label']}",
    )
    with open(TABLES_DIR / f"ece_{uq_method['label'].replace(' ', '_')}.tex", "w", encoding="utf-8") as f:
        f.write(latex_table)
    print(latex_table)

In [ ]:
for uq_method in uq_methods:
    latex_table = make_cal_metric_table_latex(
        cal_data, models, datasets_combined, uq_method=uq_method,
        prop="normalized_entropy",
        caption=f"Normalized Entropy of Bucket Counts by Model and Dataset for {uq_method['label']}",
    )
    with open(TABLES_DIR / f"normalized_entropy_{uq_method['label'].replace(' ', '_')}.tex",
              "w", encoding="utf-8") as f:
        f.write(latex_table)
    print(latex_table)

In [ ]:
for uq_method in uq_methods:
    latex_table = make_cal_metric_table_latex(
        cal_data, models, datasets_combined, uq_method=uq_method,
        prop="auroc",
        caption=f"AUROC by Model and Dataset for {uq_method['label']}",
    )
    with open(TABLES_DIR / f"auroc_{uq_method['label'].replace(' ', '_')}.tex",
              "w", encoding="utf-8") as f:
        f.write(latex_table)
    print(latex_table)

## Verbalized Uncertainty Distribution

In [ ]:
color_overrides = {
    0.0: "#C15858", 0.05: "#D57170", 0.2: "#F29999", 0.25: "#F29999",
    0.5: "#E7BA52", 0.8: "#ccdc9e", 0.85: "#ccdc9e", 0.86: "#b1c284",
    0.9: "#b1c284", 0.95: "#96a96a", 0.97: "#899D5D", 0.99: "#7c9151",
    1.0: "#637939",
}
for key in list(color_overrides.keys()):
    color_overrides[str(key)] = color_overrides[key]

scale_fonts(1.3)
fig = plot_donut_row_with_other(verbalized_stats, models, threshold_pct=5, color_dict=color_overrides)
for ext in ["svg", "pdf", "png"]:
    plt.savefig(FIGURES_DIR / f"verbalized_value_distribution_full.{ext}", bbox_inches="tight")
plt.show()

mpl.rcdefaults()
apply_matplotlib_defaults()

## P(True) Bucket Count Distribution

In [ ]:
fig = plot_bucket_distribution(ptrue_bucket_counts, models)
for ext in ["svg", "pdf", "png"]:
    plt.savefig(FIGURES_DIR / f"ptrue_bucket_counts_full.{ext}", bbox_inches="tight")
plt.show()